# 📊 Analyst Report ML Pipeline - 전체 워크플로우

이 노트북은 전체 파이프라인의 개요를 보여줍니다.

---

## 🔄 실행 순서

이 파이프라인을 순차적으로 실행하려면 아래 노트북들을 **순서대로** 실행하세요:

### **1️⃣ 단계 1: 리포트 수집 및 OCR**
📁 `01_report_collection.ipynb`
- **목표**: Naver Finance에서 리포트 크롤링 → PDF 다운로드 → 텍스트 변환
- **출력**: `data/raw/[기업명]_report_text.xlsx`
- **소요시간**: ~5-15분 (네트워크 속도에 따라)

### **2️⃣ 단계 2: Gemini LLM - 확신도 기반 점수 추출**
📁 `02_llm_confident.ipynb`
- **목표**: Gemini 2.5 Flash API로 "확신도(Confidence)"와 "강세전망(Bullish Outlook)" 점수 추출
- **프롬프트**: 0~100점 범위의 동적 점수 시스템
- **출력**: `data/processed/[기업명]_with_scores.csv`
- **비용**: API 사용료 발생 (Google Cloud 크레딧 필요)

### **3️⃣ 단계 3: 데이터 전처리 및 표준화**
📁 `03_data_preprocessing.ipynb`
- **목표**: 컬럼명 표준화, 결측값 처리, 분기별 집계
- **출력**: `data/processed/final_dataset_standardized.csv`
- **소요시간**: ~1분

### **4️⃣ 단계 4-5: 변동성 계산 및 시각화**
📁 `04_volatility_analysis.ipynb`
- **목표**: yfinance로 주가 데이터 수집 → 변동성(표준편차) 계산 → 시각화
- **출력**: `data/reports/volatility_analysis_sample.csv`, 그래프
- **소요시간**: ~2-5분

---

## 📁 디렉토리 구조

```
analyst-report-ml-pipeline/
├── notebooks/              # Jupyter 노트북들
│   ├── 00_main_workflow.ipynb      (이 파일)
│   ├── 01_report_collection.ipynb
│   ├── 02_llm_confident.ipynb
│   ├── 03_data_preprocessing.ipynb
│   └── 04_volatility_analysis.ipynb
│
├── src/                    # Python 모듈
│   ├── crawler.py          # 웹 크롤링 함수
│   ├── ocr_processor.py    # PDF → 텍스트 변환
│   ├── llm_confident.py    # Gemini API 클래스
│   ├── data_processor.py   # 데이터 전처리 함수
│   └── stock_analyzer.py   # 주가 분석 함수
│
├── data/                   # 데이터 디렉토리
│   ├── raw/                # 원본 데이터
│   ├── processed/          # 처리된 데이터
│   └── reports/            # 최종 결과 (그래프, CSV)
│
├── config.yaml             # 설정 파일
├── requirements.txt        # 의존성
└── README.md               # 프로젝트 문서
```

---

## ⚙️ 사전 준비

### 1. 환경 변수 설정

프로젝트 루트에 `.env` 파일 생성:

```bash
GEMINI_API_KEY=your_api_key_here
```

### 2. 의존성 설치

```bash
pip install -r requirements.txt
```

### 3. config.yaml 확인

- 점수 범위 (기본값: 0~100)
- 데이터 경로
- KOSPI/KOSDAQ 기업 티커

---

## 🎯 파이프라인 개요

```
[Naver Finance]
      ↓ 크롤링
[PDF 다운로드] → [OCR 처리] → [텍스트 정제]
      ↓
[01_report_collection.ipynb]
      ↓ 출력: data/raw/[기업명]_report_text.xlsx
      ↓
[Gemini API] → [확신도 + 강세전망 점수 추출]
      ↓
[02_llm_confident.ipynb]
      ↓ 출력: data/processed/[기업명]_with_scores.csv
      ↓
[데이터 정제] → [컬럼명 표준화] → [분기별 집계]
      ↓
[03_data_preprocessing.ipynb]
      ↓ 출력: data/processed/final_dataset_standardized.csv
      ↓
[주가 데이터] → [변동성 계산] → [시각화]
(yfinance)    (표준편차)       (그래프)
      ↓
[04_volatility_analysis.ipynb]
      ↓ 출력: data/reports/volatility_analysis_sample.csv
      ↓
✅ 완료!
```

---

## 📊 출력 샘플

### 최종 데이터 구조

| date | title | text | confidence_score | bullish_outlook | composite_score | year_quarter | stock_volatility |
|------|-------|------|------------------|-----------------|-----------------|--------------|------------------|
| 2024-01-15 | 삼성전자 목표가 상향 | 긍정적인... | 82 | 78 | 80 | 2024-Q1 | 0.0234 |
| 2024-02-20 | 신제품 출시 분석 | 혁신적인... | 75 | 85 | 80 | 2024-Q1 | 0.0201 |
| 2024-03-10 | 실적 전망 | 좋은 성과... | 88 | 80 | 84 | 2024-Q1 | 0.0267 |

### 생성되는 그래프

- 📈 확신도 vs 변동성 산점도
- 📊 강세전망 vs 변동성 산점도
- 🔥 상관계수 히트맵

---

## 🔑 핵심 기능

### 동적 점수 범위

`config.yaml`의 `LLM_CONFIG.score_range`를 수정하면 Gemini 프롬프트가 자동으로 조정됩니다.

```yaml
LLM_CONFIG:
  score_range:
    min: 0
    max: 100  # 이 값을 변경하면 프롬프트가 동적으로 조정됨
```

### 확신도 기반 평가

- **확신도 (Confidence)**: 분석가의 의견에 대한 확신 수준
  - 0점: 매우 불확실
  - 100점: 매우 확실

- **강세전망 (Bullish Outlook)**: 주식의 전망
  - 0점: 매우 부정적 (약세)
  - 100점: 매우 긍정적 (강세)

- **종합 점수 (Composite)**: 두 점수의 평균

---

## 📝 주의사항

1. **API 비용**: Gemini API 사용 시 비용이 발생합니다. Google Cloud 크레딧을 준비하세요.
2. **네트워크 속도**: 대량의 PDF 다운로드 시간이 오래 걸릴 수 있습니다.
3. **시스템 요구사항**: 8GB 이상의 RAM 권장
4. **API 키**: `.env` 파일에 GEMINI_API_KEY를 설정하지 않으면 프롬프트에서 입력하라고 요청됩니다.

---

## 🚀 시작하기

1. ✅ 환경 변수 설정 (`.env` 파일)
2. ✅ 의존성 설치 (`pip install -r requirements.txt`)
3. ✅ **`01_report_collection.ipynb` 실행**
4. ✅ **`02_llm_confident.ipynb` 실행**
5. ✅ **`03_data_preprocessing.ipynb` 실행**
6. ✅ **`04_volatility_analysis.ipynb` 실행**
7. 🎉 **결과 확인** (`data/reports/` 확인)

---

## 📚 참고 자료

- [Gemini API 문서](https://ai.google.dev/)
- [yfinance 문서](https://yfinance.readthedocs.io/)
- [pandas 문서](https://pandas.pydata.org/)

---

**질문이나 문제가 있으시면 README.md를 참고하세요.**